In [2]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from documents import load_pdfs,PDF_FOLDER
from chunking import chunker
from vectorstore import build_vectorstore
from retriever import retrieve
from generator import generate_answer

In [3]:
K = 3
persist_db_path = "./chroma_db"

In [4]:
def build_pipeline():
    if os.path.exists(persist_db_path):
        print("vector store found")
        embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
        db = Chroma(
            persist_directory=persist_db_path,
            embedding_function=embedding_model
        )
        return db
    
    print("vector store not found. building new")
    raw_documents = load_pdfs(PDF_FOLDER)
    print(f"[1] Loaded {len(raw_documents)} PDF(s)")

    chunks = chunker(raw_documents)
    print(f"[2] Split into {len(chunks)} chunks")

    db = build_vectorstore(chunks)
    print(f"[3] Vector store built with {len(chunks)} embedded chunks")

    return db



In [5]:
def answer_query(db, query):
    docs = retrieve(db, query, k=K)
    print(f"[4] Retrieved top {len(docs)} chunks")

    context = "\n\n".join([doc.page_content for doc in docs])
    answer = generate_answer(context, query)
    print(f"[5] Answer generated")

    return answer, docs



In [6]:
def main():
    print("Initializing system...\n")
    db = build_pipeline()
    print("\nReady. Type 'exit' to quit.\n")

    while True:
        query = input("Ask a question: ").strip()
        if not query:
            continue
        if query.lower() == "exit":
            break

        answer, sources = answer_query(db, query)

        print("\n--- ANSWER ---")
        print(answer)

        print("\n--- SOURCES USED ---")
        for i, doc in enumerate(sources):
            source_name = doc.metadata.get("source","unknown document")
            page_num = doc.metadata.get("page","N/A")
            print(f"[{i+1}] file: {source_name} | page: {page_num}")
        print("-" * 20 + "\n")



## 5 sample questions

1. "What is insulin resistance?"
2. "What are the main risk factors for type 2 diabetes?"
3. "How is type 2 diabetes diagnosed?"
4. "What medications are used to treat type 2 diabetes?"
5. "What complications can type 2 diabetes cause?"

In [ ]:
if __name__ == "__main__":
    main()

Initializing system...

vector store found


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1081.16it/s]



Ready. Type 'exit' to quit.

[4] Retrieved top 3 chunks
[5] Answer generated

--- ANSWER ---
Insulin resistance (IR) is a condition in which target cells show a reduced metabolic response to insulin, leading to a diminished ability of insulin to lower blood glucose. It can occur even when insulin levels are normal and may result from decreased insulin secretion, the presence of insulin‑antagonistic factors in the plasma, or impaired insulin signaling in target tissues.

--- SOURCES USED ---
[1] file: fnut-08-707371.pdf | page: 1
[2] file: ijms-26-01094.pdf | page: 17
[3] file: ijms-26-01094.pdf | page: 4
--------------------

[4] Retrieved top 3 chunks
[5] Answer generated

--- ANSWER ---
I cannot find that in the database.

--- SOURCES USED ---
[1] file: fendo-15-1440456.pdf | page: 17
[2] file: fendo-16-1687601.pdf | page: 2
[3] file: ijms-26-01094.pdf | page: 14
--------------------

[4] Retrieved top 3 chunks
[5] Answer generated

--- ANSWER ---
I cannot find that in the database.